In [2]:
import sys
import time
from pathlib import Path

import pandas as pd

DATA = Path.cwd().parent / "exemple_data"
DATA.mkdir(exist_ok=True)


def charger():
    """Étape 1 : crée les données brutes."""
    df = pd.DataFrame({"eleve": ["Alice", "Bob", "Chloé", "Bob"],
                       "note": [15, -3, 12, 9]})
    df.to_parquet(DATA / "brut.parquet")
    return len(df)


def nettoyer():
    """Étape 2 : lit le brut, retire les notes impossibles."""
    df = pd.read_parquet(DATA / "brut.parquet")
    df = df[df["note"] >= 0]
    df.to_parquet(DATA / "propre.parquet")
    return len(df)


def resumer():
    """Étape 3 : lit le propre, calcule la moyenne par élève."""
    df = pd.read_parquet(DATA / "propre.parquet")
    resume = df.groupby("eleve")["note"].mean().reset_index()
    resume.to_parquet(DATA / "resume.parquet")
    return len(resume)


STEPS = {
    "charger": charger,
    "nettoyer": nettoyer,
    "resumer": resumer,
}


def run(name):
    """Lance une étape, mesure sa durée, affiche le résultat."""
    start = time.perf_counter()
    lines = STEPS[name]()
    print(f"{name:<10} {lines:>4} lignes   {time.perf_counter() - start:.2f} s")


In [4]:
for name in STEPS:      # un dictionnaire garde l'ordre d'ajout
    run(name)

NameError: name 'STEPS' is not defined

In [3]:
import sys
from pathlib import Path

# Racine du projet : le dossier parent de notebooks/
PROJECT_ROOT = Path.cwd().parent.parent
PERIMETER_PATH = PROJECT_ROOT / "data" / "food_perimeter.parquet"
sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
COLS = [
    'code',
    'product_name',
    'nutriscore_grade',
    'energy-kcal_100g', 
    'fat_100g', 
    'saturated-fat_100g',
    'carbohydrates_100g', 
    'sugars_100g', 
    'fiber_100g', 
    'proteins_100g',
    'salt_100g', 
    'food_group'
]

NUTRIENTS = [
    "energy-kcal_100g",
    "fat_100g",
    "saturated-fat_100g",
    "carbohydrates_100g",
    "sugars_100g",
    "fiber_100g",
    "proteins_100g",
    "salt_100g",
]

FEATURE = "food_group"

In [8]:
import pandas as pd

In [5]:
from src.cleaning import prepare_data

In [6]:
from sklearn.model_selection import train_test_split

In [ ]:
df = prepare_data(pd.read_parquet(PERIMETER_PATH))
df = df[df['nutriscore_grade'].notna() & ~df['is_incoherent']]
df = df[COLS]
df = pd.get_dummies(df, columns=[FEATURE])

df["nutriscore_grade"] = df["nutriscore_grade"].astype('category')
df[NUTRIENTS] = df[NUTRIENTS].astype('float32')

df = df.sort_values("code").reset_index(drop=True)
df['split'] = "train" # par défaut, remplir la colonne par "train"
train_df, test_df = train_test_split(
    df.index,
    test_size=0.2,
    stratify= df['nutriscore_grade'],
    random_state=42, # la graine par défaut, permet garder le même tirage.
)
df.loc[test_df, "split"] = "test"

In [20]:
pd.crosstab(df["nutriscore_grade"], df["split"], normalize="index")


split,test,train
nutriscore_grade,,
a,0.199996,0.800004
b,0.200004,0.799996
c,0.200002,0.799998
d,0.200005,0.799995
e,0.200000,0.800000
